# Population Inference with Hierarchical Bayes 

Let's consider a non-astro application this time, a sleep-restriction study (Belenky et al. 2003).
18 subjects had their sleep restricted to a few hours a night for over a week. Each day, every subject took the same simple reaction-time test.

Subject $j$ takes a reaction time test on day $i$ of restricted sleep. So our data for subject $j$ is: 
$$y_{ij} = a_j + b_j x_{ij} + \epsilon$$
where $y_{ij}$ is the reaction time of subject $j$ on day $i$ (ms), $x_{ij}$ is the day of sleep restriction, and $\epsilon \sim \mathcal{N}(0, \sigma_{\rm obs})$. 

Everyone's reaction time slows down but not at the same rate. We're ultimately interested in learning the distribution of this rate --- i.e. population distribution of $a$ and $b$. 

For simplicity, we'll say $\sigma_{\rm obs}$ is fixed and known. We'll also assume that the distribution of $a$ and $b$ can be described using Gaussians. This means our population hyperparameters are $\phi = \{\mu_a, \sigma_a, \mu_b, \sigma_b\}$. 


In [ ]:
import time

import corner
import emcee
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import logsumexp

%matplotlib inline

Lets first draw a graphical model of the data generation process using `daft-pgm` 
> python -m pip install 'daft-pgm'


In [ ]:
import daft

In [ ]:
pgm = daft.PGM(dpi=140, node_unit=1.0, grid_unit=2.2, observed_style='shaded')

# population parameters, phi
pgm.add_node('mu_a',  r'$\mu_a$',    1.0, 4.0)
pgm.add_node('sig_a', r'$\sigma_a$', 2.2, 4.0)
pgm.add_node('mu_b',  r'$\mu_b$',    3.4, 4.0)
pgm.add_node('sig_b', r'$\sigma_b$', 4.6, 4.0)

# per-object latent parameters, theta_j = (a_j, b_j)
pgm.add_node('a', r'$a_j$', 1.6, 2.5)
pgm.add_node('b', r'$b_j$', 4.0, 2.5)

# the data, and the quantities held fixed
pgm.add_node('y',    r'$y_{ij}$',           2.9, 1.1, observed=True)
pgm.add_node('x',    r'$x_{ij}$',           1.5, 1.1, fixed=True)
pgm.add_node('sobs', r'$\sigma_{\rm obs}$', 5.6, 1.1, fixed=True)

# one edge per conditional dependence in the joint
for parent, child in [('mu_a', 'a'), ('sig_a', 'a'), ('mu_b', 'b'), ('sig_b', 'b'),
                      ('a', 'y'), ('b', 'y'), ('x', 'y'), ('sobs', 'y')]:
    pgm.add_edge(parent, child)

# plates: OUTERMOST FIRST -- daft fills each plate white, so a plate added
# later paints over one added earlier
pgm.add_plate([0.55, 0.1, 4.0, 3.15], label=r'$j = 1,\dots,J$',
              shift=-0.12, position='bottom right')
pgm.add_plate([0.9, 0.45, 2.6, 1.3], label=r'$i = 1,\dots,n_j$',
              shift=-0.12, position='bottom left')

pgm.render()

Lets generate some mock data hierarchically based on this graph

In [ ]:
np.random.seed(42)

J = 18                       # subjects
MU_A, SIG_A = 250.0, 25.0    # baseline reaction time: mean and population scatter (ms)
MU_B, SIG_B = 10.0, 7.0      # decline per day:        mean and population scatter (ms/day)
SIG_OBS = 30.0               # known measurement scatter (ms)
PHI_TRUE = np.array([MU_A, MU_B, SIG_A, SIG_B])

# draws each subject's parameters from the population
a_true = np.random.normal(MU_A, SIG_A, size=J)
b_true = np.random.normal(MU_B, SIG_B, size=J)

# "conduct the experiment": data collected 3 to 5 days per subject, sparsely sampled
n_j = np.random.randint(3, 6, J)
x_list, y_list = [], []
for j in range(J):
    x_j = np.sort(np.random.choice(np.arange(10), size=n_j[j], replace=False)).astype(float)
    x_list.append(x_j)
    y_list.append(a_true[j] + b_true[j] * x_j + np.random.normal(0, SIG_OBS, n_j[j]))

print(f'{J} subjects, {n_j.sum()} measurements total, {n_j.min()}-{n_j.max()} days each')
print(f'population truth:  mu_a={MU_A:.0f}  mu_b={MU_B:.0f}  sigma_a={SIG_A:.0f}  sigma_b={SIG_B:.0f}')
print()
print('This particular sample of J subjects (what a perfect-measurement experiment would find):')
print(f'  mean a_j = {a_true.mean():7.2f}   sd a_j = {a_true.std(ddof=1):6.2f}')
print(f'  mean b_j = {b_true.mean():7.2f}   sd b_j = {b_true.std(ddof=1):6.2f}')

plot the data 

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4.4))
cmap = plt.cm.viridis(np.linspace(0, 0.9, J))
for j in range(J):
    ax.errorbar(x_list[j], y_list[j], yerr=SIG_OBS, fmt='o-', ms=3.5, lw=0.9,
                color=cmap[j], alpha=0.75)
ax.set_xlabel('day of sleep restriction')
ax.set_ylabel('reaction time (ms)')
plt.tight_layout(); plt.show()

## derive individual posteriors with `emcee`

Fit each subject individually, with a flat prior on $\theta_j = (a_j, b_j)$. Sample the posterior $p(\theta_j| \{x_{ij}, y_{ij}\})$. Assume a Gaussian likelihood so you can write down the log posterior as: 

$$\log p(\theta_j \mid y_j) = -\frac{1}{2}\sum_{i=1}^{n_j}
  \left(\frac{y_{ij} - a_j - b_j x_{ij}}{\sigma_{\rm obs}}\right)^{\!2} + \text{const}$$

Store the chains for all the subject

In [ ]:
def log_theta_prior(theta): 
    # flat prior on theta
    a, b = theta
    if not (0.0 < a < 600.0 and -100.0 < b < 100.0):
        return -np.inf
    else: 
        return 0.

def log_likelihood_j(theta, x, y): 
    # implement the likelihood above


def log_posterior_j(theta, x, y):
    ''' log posterior for one subject
    '''
    # combine the prior and likelihood here

In [ ]:
# feel free to changes these numbers
NWALKERS, NSTEPS, NBURN = 32, 3000, 600

chains = [] 
for j in range(J): # loop through each of the subjects

    # initialize the walkers


    # run emcee.EnsembleSampler to sample log_posterior_j
    sampler = 

    # save chains after burn-in and thinning
    chains.append(sampler.get_chain(discard=NBURN, thin=15), flat=True))

## incorrect population inference
Let's intentionally do the wrong population inference. Instead of hierarchical bayes, let's do something "sensible" and histogram the median values of each posterior $p(\theta_j|\{x_{ij}, y_{ij}\}$, as is common done in the literature. 

1. get the median $(a, b)$ values of the chains
2. calculate the mean and standard deviation of the medians
3. compare them to the true population hyperparameters $(\mu_a, \sigma_a, \mu_b, \sigma_b)$

## Correct population inference
We went over this in class but lets go over it again so yo have it in front of you. 

Start from the joint density of the hierarchical model,

$$p(\phi, \theta_{1:J}, y) \;=\; p(\phi) \prod_{j=1}^{J} p(\theta_j \mid \phi)\, p(y_j \mid \theta_j)$$


We want $p(\phi \mid y)$, so we can derive:

$$p(\phi \mid y) \;=\; \int p(\phi, \theta_{1:J} \mid y)\, d\theta_{1:J}
\;=\; \frac{p(\phi)}{p(y)} \int \prod_{j=1}^{J}
\Big[ p(\theta_j \mid \phi)\, p(y_j \mid \theta_j) \Big]\, d\theta_1 \cdots d\theta_J$$

The integrand is a product of terms each depending on a single $\theta_j$. Since $\theta_j$ and $\theta_k$ are conditional independent, the integral factorizes term by term:

$$\boxed{\;p(\phi \mid y) \;\propto\; p(\phi) \prod_{j=1}^{J} p(y_j \mid \phi), \qquad
p(y_j \mid \phi) \;\equiv\; \int p(y_j \mid \theta)\, p(\theta \mid \phi)\, d\theta \;}$$

$p(y_j\mid\phi)$ is the **population likelihood**, which we can evaluate using the individual posterior chains you saved above. Since we used flat prior for $p(\theta)$, 
$p(y_j \mid \theta) \propto p(\theta \mid y_j)$, so 

$$p(y_j \mid \phi) \;\propto\; \int p(\theta \mid y_j)\, p(\theta \mid \phi)\, d\theta
\;\approx\; \frac{1}{S}\sum_{s=1}^{S} p(\theta_{js} \mid \phi),
\qquad \theta_{js} \sim p(\theta \mid y_j)$$


Working in log space, the population log posterior is
$$\log p(\phi \mid y) = \log p(\phi) + \sum_{j=1}^{J} \log \sum\limits_s p(\theta_{js}\mid\mu_a,\sigma_a) + \text{const}$$
we can use `logsumexp` for convenience 
$$\log p(\phi \mid y) = \log p(\phi) + \sum_{j=1}^{J} \operatorname*{logsumexp}_{s} \Big[ \log p(\theta_{js}\mid\mu_a,\sigma_a) \Big] + \text{const}$$

In [ ]:
# randomly pick the same number of samples S from the chains. 
S = 1000

pick = [np.random.choice(len(c), S, replace=False) for c in chains]
A_S = np.array([chains[j][pick[j], 0] for j in range(J)])   # (J, S) samples of a_j
B_S = np.array([chains[j][pick[j], 1] for j in range(J)])   # (J, S) samples of b_j


def log_prior_phi(phi):
    # simple flat prior on hyperparameters
    mu_a, mu_b, sigma_a, sigma_b = phi
    ok = (0 < mu_a < 600) and (-100 < mu_b < 100) and (0 < sigma_a < 200) and (0 < sigma_b < 60)
    return 0.0 if ok else -np.inf


def log_like_population(phi):
    ''' Monte-Carlo estimate of sum_j log p(y_j | phi) from above
    '''
    mu_a, mu_b, sigma_a, sigma_b = phi
    log_w = (-0.5 * ((A_S - mu_a) / sigma_a) ** 2 - np.log(sigma_a)
             - 0.5 * ((B_S - mu_b) / sigma_b) ** 2 - np.log(sigma_b))   
    return np.sum(logsumexp(log_w, axis=1))     # constants dropped


def log_posterior_phi(phi):
    lp = log_prior_phi(phi)
    if not np.isfinite(lp): 
        return -np.inf 
    else:
        return lp + log_like_population(phi)

## run `emcee` on the population hyperparameter posterior

In [ ]:
# initialize mu_a, sigma_a, mu_b, sigma_b


# run EnsembleSampler

plot the posterior using `corner` and compare it to the true hyperparameters